# FaceEncoder Model

ResNet50 backbone with a linear projection head. `forward()` returns raw embeddings; `encode()` returns L2-normalized embeddings for inference.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

## Model Definition

In [2]:
class FaceEncoder(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone.fc = nn.Linear(2048, embedding_dim)
        self._embedding_dim = embedding_dim

    @property
    def embedding_dim(self):
        return self._embedding_dim

    def forward(self, x):
        return self.backbone(x)

    def encode(self, x):
        return F.normalize(self.forward(x), p=2, dim=1)

## Sanity Check

Pass a random batch through `.encode()` and verify output shape and unit norm.

In [3]:
model = FaceEncoder(512)
model.eval()
x = torch.randn(2, 3, 112, 112)
with torch.inference_mode():
    out = model.encode(x)
print("Output shape:", out.shape)          # (2, 512)
print("L2 norms:", out.norm(dim=1))        # should be ~1.0

Output shape: torch.Size([2, 512])
L2 norms: tensor([1.0000, 1.0000])
